In [1]:
import pandas as pd
import numpy as np
import math
import networkx as nx
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [2]:
def minkowski_distance(x, y, p):
    distance = sum(abs(xi - yi) ** p for xi, yi in zip(x, y)) ** (1 / p)
    return distance

def find_maximal_cliques(adj):
    # 创建图
    G = nx.Graph(adj)
    # 找到所有极大团
    cliques = list(nx.find_cliques(G))
    return cliques

def

In [3]:
data = pd.read_csv('glass.csv')
#data = pd.read_csv('sonar.csv',header=None)
print(data.head)

y = data.iloc[:, -1].to_numpy()
scaler = MinMaxScaler()
X = scaler.fit_transform(data.iloc[:, :-1])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
n, m = X_train.shape

<bound method NDFrame.head of           RI     Na    Mg    Al     Si     K    Ca    Ba   Fe  Type
0    1.52101  13.64  4.49  1.10  71.78  0.06  8.75  0.00  0.0     1
1    1.51761  13.89  3.60  1.36  72.73  0.48  7.83  0.00  0.0     1
2    1.51618  13.53  3.55  1.54  72.99  0.39  7.78  0.00  0.0     1
3    1.51766  13.21  3.69  1.29  72.61  0.57  8.22  0.00  0.0     1
4    1.51742  13.27  3.62  1.24  73.08  0.55  8.07  0.00  0.0     1
..       ...    ...   ...   ...    ...   ...   ...   ...  ...   ...
209  1.51623  14.14  0.00  2.88  72.61  0.08  9.18  1.06  0.0     7
210  1.51685  14.92  0.00  1.99  73.06  0.00  8.40  1.59  0.0     7
211  1.52065  14.36  0.00  2.02  73.42  0.00  8.44  1.64  0.0     7
212  1.51651  14.38  0.00  1.94  73.61  0.00  8.48  1.57  0.0     7
213  1.51711  14.23  0.00  2.08  73.36  0.00  8.62  1.67  0.0     7

[214 rows x 10 columns]>
(171, 9)


In [5]:
print('# of test:', len(y_test))

delta = 0.2 * math.sqrt(m)
alpha = 0.7

print('delta: ', delta, 'alpha: ', alpha)

dis_arr = np.zeros((n,n))
neighborhoods = []
neighborhoods_label = []

for i in range(n):
    neighbors = []
    temp_label = []
    for j in range(n):
        dis_arr[i][j] = minkowski_distance(X_train[i], X_train[j], 2)
        if dis_arr[i][j] < delta:
            neighbors.append(j)
            temp_label.append(y_train[j])
    
    count = pd.Series(temp_label).value_counts()
    if not count[count > alpha * len(neighbors)].empty:
        neighborhoods.append(i)
        neighborhoods_label.append(count[count > alpha * len(neighbors)].index.tolist()[0])

print('Total: ', n, 'Filtered: ', len(neighborhoods))

for i, t in enumerate(X_test):
    mc_labels = []
    for j in neighborhoods:
        ob = X_train[j]
        if minkowski_distance(t, ob, 2) < delta:
            mc_labels.append(y_train[j])
    if mc_labels:
        percentage = pd.Series(mc_labels).value_counts(normalize=True) * 100
        res = ', '.join([f"{element}: {pct:.2f}%" for element, pct in percentage.items()])
    else:
        res = ''
    print(y_test[i], res)

# of test: 43
delta:  0.6000000000000001 alpha:  0.7
Total:  171 Filtered:  25
1 
7 7: 100.00%
1 2: 100.00%
7 7: 100.00%
2 7: 100.00%
2 2: 100.00%
1 
2 7: 100.00%
2 7: 100.00%
2 
6 7: 93.33%, 2: 6.67%
5 2: 100.00%
2 
2 7: 100.00%
6 
5 7: 100.00%
7 7: 100.00%
1 2: 100.00%
1 2: 100.00%
6 
2 7: 100.00%
7 7: 100.00%
7 7: 100.00%
7 7: 100.00%
3 
2 2: 100.00%
1 
1 
5 2: 100.00%
1 
1 
2 7: 100.00%
3 
2 7: 100.00%
1 2: 100.00%
7 7: 100.00%
5 5: 100.00%
3 
2 
2 7: 100.00%
2 
7 7: 100.00%
1 


In [4]:
print('# of test:', len(y_test))

delta = 0.2 * math.sqrt(m)
alpha = 0.7

print('delta: ', delta, 'alpha: ', alpha)

adj_arr = np.zeros((n,n))
dis_arr = np.zeros((n,n))

for i in range(n):
    for j in range(i, n):
        dis_arr[i][j] = minkowski_distance(X_train[i], X_train[j], 2)
        dis_arr[j][i] = dis_arr[i][j]
        if dis_arr[i][j] < delta:
            adj_arr[i][j] = 1
            adj_arr[j][i] = 1
            
cliques = find_maximal_cliques(adj_arr)

filtered_cliques = []
cliques_label = []
for clique in cliques:
    temp_label = []
    for j in clique:
        temp_label.append(y_train[j])
    
    count = pd.Series(temp_label).value_counts()
    if not count[count > alpha * len(clique)].empty:
        filtered_cliques.append(clique)
        cliques_label.append(count[count > alpha * len(clique)].index.tolist()[0])
        
print('Total: ', len(cliques), 'Filtered: ', len(filtered_cliques))

for i, t in enumerate(X_test):
    t_pos = 0;
    t_neg = 0;
    cy = []
    mc_labels = []
    for k, clique in enumerate(filtered_cliques):
        flag = True
        for j in clique:
            if minkowski_distance(t, X_train[j], 2) >= delta:
                flag = False
                break
        if flag:
            cy.append(clique)
            mc_labels.append(cliques_label[k])
    if mc_labels:
        percentage = pd.Series(mc_labels).value_counts(normalize=True) * 100
        res = ', '.join([f"{element}: {pct:.2f}%" for element, pct in percentage.items()])
    else:
        res = ''
    print(y_test[i], res)

# of test: 43
delta:  0.6000000000000001 alpha:  0.7
Total:  226 Filtered:  17
1 
7 7: 100.00%
1 1: 100.00%
7 7: 100.00%
2 2: 100.00%
2 
1 
2 2: 100.00%
2 2: 100.00%
2 
6 7: 100.00%
5 2: 100.00%
2 
2 2: 100.00%
6 
5 
7 7: 80.00%, 2: 20.00%
1 1: 100.00%
1 1: 100.00%
6 
2 2: 100.00%
7 7: 100.00%
7 7: 100.00%
7 7: 100.00%
3 
2 1: 100.00%
1 
1 
5 2: 100.00%
1 
1 
2 2: 100.00%
3 
2 2: 100.00%
1 1: 100.00%
7 7: 100.00%
5 5: 100.00%
3 
2 
2 2: 50.00%, 7: 50.00%
2 
7 7: 100.00%
1 
